# Convergence and Robustness of the Pathway Classification

This notebook checks that the primary-label classification produced by
[`01_pathway_classification.ipynb`](01_pathway_classification.ipynb) is
stable and not an artifact of the number of sampled pathways or the exact
quantile thresholds used.

It expects as input the `paths_order_descriptors_with_archetype_labels.csv`
and `paths_with_primary_labels.csv` files produced by that notebook, and
performs:

1. **Class fingerprints** -- median effective-link profile per primary label.
2. **Subsampling convergence** -- re-classifying random subsets of the
   pathway ensemble to check that class fractions stabilize as more
   pathways are included.
3. **Threshold convergence (SI)** -- convergence of the quantile-based
   thresholds themselves.
4. **Bootstrap uncertainty (SI)** -- shrinking confidence intervals with
   increasing sample size.
5. **Threshold sensitivity (SI)** -- re-classifying under perturbed
   quantile cutoffs to check label agreement with the default thresholds.


## 1. Configuration

In [ ]:
ROOT_DIR = Path("./example_data")  # TODO: point this at a directory containing path_couplings_timeseries.csv files for one GB structure

PRIMARY_LABEL_FILE = ROOT_DIR / "paths_with_primary_labels.csv"
ORDER_DESCRIPTOR_FILE = ROOT_DIR / "paths_order_descriptors_with_archetype_labels.csv"
REPRESENTATIVE_FILE = ROOT_DIR / "representative_paths_for_dynamics.csv"

# Optional mobility file.
# Leave as None if not available.
# Accepted formats:
#   1) columns: path, mobility
#   2) columns: label_primary, mobility
MOBILITY_FILE = None
# MOBILITY_FILE = ROOT_DIR / "pathway_mobility.csv"

OUTPUT_DIR = ROOT_DIR / "classification_reliability_figures"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 123
N_BOOT = 300
N_MIN = 30
N_STEP = 25

GRID_SIZE = 100
NORMALIZATION = "median"   # options: "raw", "median", "max", "first_last"

PLOT_MEDOID_IN_FINGERPRINT = False

INCLUDE_DIFFUSE_AS_5TH_LABEL = True
FLAT_COUNTS_AS_NON_RECOVERING = True

# If your GB is approximately in the middle of the normalized path:
GB_REGION = (0.40, 0.60)

LABEL_ORDER = [
    "Source-limited",
    "GB-crossing (recovering)",
    "GB-blocked (non-recovering)",
    "Target-limited",
    "Diffuse/weakly structured",
]

## 2. Classification functions

Same quantile-threshold classification logic as the main pipeline notebook, exposed with adjustable quantile levels so it can be reused for threshold-sensitivity checks.

In [ ]:
def label_archetypes_from_descriptors(
    desc_df,
    q_source=20,
    q_target=80,
    q_high=75,
    q_low=25,
    rec_hi=1.05,
    rec_lo=0.95,
):
    """
    Assign order-aware labels using quantile thresholds recomputed
    from the provided dataframe.
    """

    out = desc_df.copy()

    required_cols = [
        "pE", "E_dip", "C_dip", "rE", "rJ", "rC",
        "Lweak_E", "Lweak_C",
        "drop_E", "rise_E", "drop_C", "rise_C", "RE"
    ]

    missing = [c for c in required_cols if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required descriptor columns: {missing}")

    pE_lo = float(np.percentile(out["pE"], q_source))
    pE_hi = float(np.percentile(out["pE"], q_target))

    E_dip_q = float(np.percentile(out["E_dip"], q_high))
    C_dip_q = float(np.percentile(out["C_dip"], q_high))

    rE_q = float(np.percentile(out["rE"], q_low))
    rJ_q = float(np.percentile(out["rJ"], q_low))
    rC_q = float(np.percentile(out["rC"], q_low))

    Lweak_E_q = float(np.percentile(out["Lweak_E"], q_high))
    Lweak_C_q = float(np.percentile(out["Lweak_C"], q_high))

    drop_E_q = float(np.percentile(out["drop_E"], q_high))
    rise_E_q = float(np.percentile(out["rise_E"], q_high))
    drop_C_q = float(np.percentile(out["drop_C"], q_high))
    rise_C_q = float(np.percentile(out["rise_C"], q_high))

    label_locations = []
    label_causes = []
    label_shapes = []
    label_recoveries = []
    label_combined = []

    for _, r in out.iterrows():

        # Location
        if r["pE"] <= pE_lo:
            location = "source-limited"
        elif r["pE"] >= pE_hi:
            location = "target-limited"
        else:
            location = "GB-limited"

        # Cause
        coupling_signal = (
            (r["E_dip"] >= E_dip_q)
            or (r["drop_E"] >= drop_E_q)
            or (r["Lweak_E"] >= Lweak_E_q)
            or (r["rJ"] <= rJ_q)
        )

        coherence_signal = (
            (r["C_dip"] >= C_dip_q)
            or (r["drop_C"] >= drop_C_q)
            or (r["Lweak_C"] >= Lweak_C_q)
            or (r["rC"] <= rC_q)
        )

        if coupling_signal and not coherence_signal:
            cause = "coupling-limited"
        elif coherence_signal and not coupling_signal:
            cause = "coherence-limited"
        else:
            cause = "mixed-limited"

        # Shape
        sharp = (
            ((r["drop_E"] >= drop_E_q) and (r["rise_E"] >= rise_E_q))
            or ((r["drop_C"] >= drop_C_q) and (r["rise_C"] >= rise_C_q))
        )

        extended = (
            (r["Lweak_E"] >= Lweak_E_q)
            or (r["Lweak_C"] >= Lweak_C_q)
        )

        if sharp and extended:
            shape = "sharp+extended"
        elif sharp:
            shape = "sharp"
        elif extended:
            shape = "extended"
        else:
            shape = "diffuse"

        # Recovery
        if r["RE"] > rec_hi:
            recovery = "recovering"
        elif r["RE"] < rec_lo:
            recovery = "degrading"
        else:
            recovery = "flat"

        combined = f"{location}, {cause}, {shape}, {recovery}"

        label_locations.append(location)
        label_causes.append(cause)
        label_shapes.append(shape)
        label_recoveries.append(recovery)
        label_combined.append(combined)

    out["label_location"] = label_locations
    out["label_cause"] = label_causes
    out["label_shape"] = label_shapes
    out["label_recovery"] = label_recoveries
    out["label_combined"] = label_combined

    thresholds = {
        "pE_q_source": pE_lo,
        "pE_q_target": pE_hi,
        "E_dip_q_high": E_dip_q,
        "C_dip_q_high": C_dip_q,
        "rE_q_low": rE_q,
        "rJ_q_low": rJ_q,
        "rC_q_low": rC_q,
        "Lweak_E_q_high": Lweak_E_q,
        "Lweak_C_q_high": Lweak_C_q,
        "drop_E_q_high": drop_E_q,
        "rise_E_q_high": rise_E_q,
        "drop_C_q_high": drop_C_q,
        "rise_C_q_high": rise_C_q,
        "recovery_hi": rec_hi,
        "recovery_lo": rec_lo,
        "q_source": q_source,
        "q_target": q_target,
        "q_high": q_high,
        "q_low": q_low,
    }

    return out, thresholds


def add_primary_label(
    df,
    include_diffuse_as_5th=True,
    flat_counts_as_nonrecovering=True,
):
    """
    Merge detailed labels into primary transport classes.
    """

    out = df.copy()

    if flat_counts_as_nonrecovering:
        nonrecovering = out["label_recovery"].isin(["degrading", "flat"])
    else:
        nonrecovering = out["label_recovery"].eq("degrading")

    out["label_primary"] = "Unassigned"

    if include_diffuse_as_5th:
        out.loc[out["label_shape"].eq("diffuse"), "label_primary"] = "Diffuse/weakly structured"

    out.loc[out["label_location"].eq("source-limited"), "label_primary"] = "Source-limited"
    out.loc[out["label_location"].eq("target-limited"), "label_primary"] = "Target-limited"

    gb = out["label_location"].eq("GB-limited")

    out.loc[
        gb & out["label_recovery"].eq("recovering"),
        "label_primary"
    ] = "GB-crossing (recovering)"

    out.loc[
        gb & nonrecovering,
        "label_primary"
    ] = "GB-blocked (non-recovering)"

    return out


def classify_dataframe(
    df,
    q_source=20,
    q_target=80,
    q_high=75,
    q_low=25,
):
    """
    Full classification wrapper.
    """

    labeled, thresholds = label_archetypes_from_descriptors(
        df,
        q_source=q_source,
        q_target=q_target,
        q_high=q_high,
        q_low=q_low,
    )

    labeled = add_primary_label(
        labeled,
        include_diffuse_as_5th=INCLUDE_DIFFUSE_AS_5TH_LABEL,
        flat_counts_as_nonrecovering=FLAT_COUNTS_AS_NON_RECOVERING,
    )

    return labeled, thresholds

## 3. Plotting helpers

In [ ]:
def ordered_present_labels(labels):
    labels = list(pd.Series(labels).dropna().unique())
    ordered = [x for x in LABEL_ORDER if x in labels]
    for x in labels:
        if x not in ordered:
            ordered.append(x)
    return ordered


def savefig(fig, name):
    png = OUTPUT_DIR / f"{name}.png"
    pdf = OUTPUT_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved:\n  {png}\n  {pdf}")

## 4. Class-wise fingerprint plot

Median (with IQR) of the normalized effective-link profile $E_i = J_i C_i$ along the pathway, grouped by primary label.

In [ ]:
def get_jce_file(path_string):
    return Path(path_string).with_suffix(".JCE_vectors.csv")


def load_E_profile(path_string):
    """
    Load E_i profile from saved JCE vectors.
    Works with columns:
      J, C, E
    or
      J_abs_mean, C_coherence, E_JxC
    """

    jce_file = get_jce_file(path_string)

    if not jce_file.exists():
        raise FileNotFoundError(f"JCE file not found: {jce_file}")

    df = pd.read_csv(jce_file)
    df.columns = [c.strip() for c in df.columns]

    if "E" in df.columns:
        E = df["E"].to_numpy(float)
    elif "E_JxC" in df.columns:
        E = df["E_JxC"].to_numpy(float)
    elif "J" in df.columns and "C" in df.columns:
        E = df["J"].to_numpy(float) * df["C"].to_numpy(float)
    elif "J_abs_mean" in df.columns and "C_coherence" in df.columns:
        E = df["J_abs_mean"].to_numpy(float) * df["C_coherence"].to_numpy(float)
    else:
        raise ValueError(
            f"Cannot find E or J/C columns in {jce_file}. "
            f"Available columns: {list(df.columns)}"
        )

    if len(E) == 1:
        x = np.array([0.5])
    else:
        x = np.linspace(0.0, 1.0, len(E))

    return x, E


def normalize_profile(E, method="median"):
    E = np.asarray(E, dtype=float)

    if method == "raw":
        return E
    elif method == "median":
        scale = np.median(E)
    elif method == "max":
        scale = np.max(E)
    elif method == "first_last":
        scale = 0.5 * (E[0] + E[-1])
    else:
        raise ValueError(f"Unknown normalization: {method}")

    if scale <= 0 or not np.isfinite(scale):
        return E

    return E / scale


def interpolate_profile(x, y, grid):
    if len(x) == 1:
        return np.ones_like(grid) * y[0]
    return np.interp(grid, x, y)


def collect_profiles_for_label(df_primary, label, grid):
    sub = df_primary[df_primary["label_primary"] == label].copy()

    profiles = []
    skipped = []

    for _, row in sub.iterrows():
        try:
            x, E = load_E_profile(row["path"])
            E_norm = normalize_profile(E, method=NORMALIZATION)
            E_interp = interpolate_profile(x, E_norm, grid)
            profiles.append(E_interp)
        except Exception as err:
            skipped.append((row["path"], str(err)))

    if len(skipped) > 0:
        print(f"{label}: skipped {len(skipped)} paths.")
        for p, err in skipped[:3]:
            print(p)
            print(err)

    if len(profiles) == 0:
        return None

    return np.vstack(profiles)


def plot_class_fingerprints():
    df_primary = pd.read_csv(PRIMARY_LABEL_FILE)

    labels = ordered_present_labels(df_primary["label_primary"])
    grid = np.linspace(0.0, 1.0, GRID_SIZE)

    rep_df = None
    if PLOT_MEDOID_IN_FINGERPRINT and REPRESENTATIVE_FILE.exists():
        rep_df = pd.read_csv(REPRESENTATIVE_FILE)

    fig, axes = plt.subplots(
        len(labels),
        1,
        figsize=(7.2, 2.6 * len(labels)),
        sharex=True,
    )

    if len(labels) == 1:
        axes = [axes]

    rows = []

    for ax, label in zip(axes, labels):

        profiles = collect_profiles_for_label(df_primary, label, grid)

        if profiles is None:
            ax.text(0.5, 0.5, f"No profiles for {label}", ha="center", va="center")
            continue

        med = np.median(profiles, axis=0)
        q25 = np.percentile(profiles, 25, axis=0)
        q75 = np.percentile(profiles, 75, axis=0)

        ax.axvspan(GB_REGION[0], GB_REGION[1], alpha=0.12, label="GB region")

        ax.plot(grid, med, linewidth=2.3, label="Median")
        ax.fill_between(grid, q25, q75, alpha=0.25, label="IQR")

        if rep_df is not None:
            rep_rows = rep_df[rep_df["label_primary"] == label]
            if len(rep_rows) > 0:
                try:
                    x_rep, E_rep = load_E_profile(rep_rows.iloc[0]["path"])
                    E_rep = normalize_profile(E_rep, method=NORMALIZATION)
                    E_rep = interpolate_profile(x_rep, E_rep, grid)
                    ax.plot(grid, E_rep, linestyle="--", linewidth=1.8, label="Medoid")
                except Exception as err:
                    print(f"Could not plot medoid for {label}: {err}")

        ax.set_title(f"{label}  (N = {profiles.shape[0]})")
        ax.set_ylabel(r"$E_i / \mathrm{median}(E_i)$" if NORMALIZATION == "median" else r"$E_i$")
        ax.grid(alpha=0.3)
        ax.legend(frameon=False, fontsize=8)

        for x, m, lo, hi in zip(grid, med, q25, q75):
            rows.append({
                "label_primary": label,
                "x_norm": x,
                "median": m,
                "q25": lo,
                "q75": hi,
                "n_profiles": profiles.shape[0],
                "normalization": NORMALIZATION,
            })

    axes[-1].set_xlabel("Normalized pathway position")

    fig.suptitle(
        r"Class-wise fingerprints of effective link strength $E_i = J_i C_i$",
        fontsize=13,
        y=1.01,
    )

    plt.tight_layout()

    out_csv = OUTPUT_DIR / "MAIN_class_fingerprint_E_profiles_data.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"Saved data: {out_csv}")

    savefig(fig, "MAIN_class_fingerprint_E_profiles")
    plt.show()

## 5. Subsampling convergence

Repeatedly re-classifies random subsets of the sampled pathways (recomputing quantile thresholds each time) to test whether the primary-label class fractions have stabilized with respect to the number of sampled pathways.

In [ ]:
def run_subsampling_convergence():
    df_all = pd.read_csv(ORDER_DESCRIPTOR_FILE)

    df_full, full_thresholds = classify_dataframe(df_all)
    labels = ordered_present_labels(df_full["label_primary"])

    print("\nFull dataset label counts:")
    print(df_full["label_primary"].value_counts())

    rng = np.random.default_rng(RANDOM_SEED)
    n_total = len(df_all)

    n_values = list(range(N_MIN, n_total + 1, N_STEP))
    if n_values[-1] != n_total:
        n_values.append(n_total)

    results = []

    threshold_keys = [
        "pE_q_source",
        "pE_q_target",
        "E_dip_q_high",
        "C_dip_q_high",
        "rE_q_low",
        "Lweak_E_q_high",
        "drop_E_q_high",
        "rise_E_q_high",
    ]

    for n_sample in n_values:
        print(f"Subsampling N = {n_sample}")

        for boot in range(N_BOOT):

            idx = rng.choice(n_total, size=n_sample, replace=False)
            df_sub = df_all.iloc[idx].copy()

            df_lab, thresholds = classify_dataframe(df_sub)

            base = {
                "n_sample": n_sample,
                "bootstrap": boot,
                "n_total": len(df_lab),
                "n_GB_limited": int((df_lab["label_location"] == "GB-limited").sum()),
            }

            # Primary fractions
            frac = df_lab["label_primary"].value_counts(normalize=True)
            for lab in labels:
                row = base.copy()
                row["quantity"] = "primary_fraction"
                row["label"] = lab
                row["value"] = float(frac.get(lab, 0.0))
                results.append(row)

            # GB-only split
            df_gb = df_lab[df_lab["label_location"] == "GB-limited"].copy()
            if len(df_gb) > 0:
                gb_frac = df_gb["label_primary"].value_counts(normalize=True)
                for lab in ["GB-crossing (recovering)", "GB-blocked (non-recovering)"]:
                    row = base.copy()
                    row["quantity"] = "GB_fraction"
                    row["label"] = lab
                    row["value"] = float(gb_frac.get(lab, 0.0))
                    results.append(row)

            # Thresholds
            for key in threshold_keys:
                row = base.copy()
                row["quantity"] = "threshold"
                row["label"] = key
                row["value"] = float(thresholds[key])
                results.append(row)

    df_conv = pd.DataFrame(results)

    out_csv = OUTPUT_DIR / "subsampling_convergence_results.csv"
    df_conv.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

    full_frac = df_full["label_primary"].value_counts(normalize=True)
    df_full_gb = df_full[df_full["label_location"] == "GB-limited"].copy()
    full_gb_frac = df_full_gb["label_primary"].value_counts(normalize=True)

    return df_conv, df_full, labels, full_frac, full_gb_frac


def summarize_bootstrap(df, quantity):
    sub = df[df["quantity"] == quantity].copy()

    summary = (
        sub.groupby(["n_sample", "label"])["value"]
        .agg(
            mean="mean",
            q025=lambda x: np.percentile(x, 2.5),
            q975=lambda x: np.percentile(x, 97.5),
            std="std",
        )
        .reset_index()
    )
    return summary


def plot_main_subsampling_convergence(df_conv, labels, full_frac, full_gb_frac):
    summary_primary = summarize_bootstrap(df_conv, "primary_fraction")
    summary_gb = summarize_bootstrap(df_conv, "GB_fraction")

    fig, axes = plt.subplots(2, 1, figsize=(7.2, 7.2), sharex=False)

    # Panel A
    ax = axes[0]

    for lab in labels:
        sub = summary_primary[summary_primary["label"] == lab]
        if len(sub) == 0:
            continue

        ax.plot(sub["n_sample"], sub["mean"], linewidth=2, label=lab)
        ax.fill_between(sub["n_sample"], sub["q025"], sub["q975"], alpha=0.20)

        if lab in full_frac.index:
            ax.axhline(full_frac[lab], linestyle="--", linewidth=1, alpha=0.35)

    ax.set_title("Subsampling convergence of primary pathway classes")
    ax.set_ylabel("Class fraction")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8, ncol=2)

    # Panel B
    ax = axes[1]

    for lab in ["GB-crossing (recovering)", "GB-blocked (non-recovering)"]:
        sub = summary_gb[summary_gb["label"] == lab]
        if len(sub) == 0:
            continue

        ax.plot(sub["n_sample"], sub["mean"], linewidth=2, label=lab)
        ax.fill_between(sub["n_sample"], sub["q025"], sub["q975"], alpha=0.20)

        if lab in full_gb_frac.index:
            ax.axhline(full_gb_frac[lab], linestyle="--", linewidth=1, alpha=0.35)

    ax.set_title("Subsampling convergence of GB recovery behavior")
    ax.set_xlabel("Number of sampled pathways")
    ax.set_ylabel("Fraction within GB-limited pathways")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)

    plt.tight_layout()
    savefig(fig, "MAIN_subsampling_convergence")
    plt.show()

## 6. Class-weighted mobility convergence (optional)

Only runs if a mobility CSV is supplied via `MOBILITY_FILE`.

In [ ]:
def plot_weighted_mobility_convergence(df_conv, df_full):
    """
    This function supports two mobility file formats.

    Format A: per-path mobility
        columns: path, mobility

    Format B: per-class mobility
        columns: label_primary, mobility
    """

    if MOBILITY_FILE is None:
        print("\nSkipping weighted mobility convergence: MOBILITY_FILE = None")
        return

    mob = pd.read_csv(MOBILITY_FILE)

    rng = np.random.default_rng(RANDOM_SEED)
    df_all = pd.read_csv(ORDER_DESCRIPTOR_FILE)
    n_total = len(df_all)

    n_values = sorted(df_conv["n_sample"].unique())

    rows = []

    if {"path", "mobility"}.issubset(mob.columns):
        print("Using per-path mobility file.")

        df_all_with_mob = df_all.merge(mob[["path", "mobility"]], on="path", how="left")

        for n_sample in n_values:
            print(f"Mobility convergence N = {n_sample}")

            for boot in range(N_BOOT):
                idx = rng.choice(n_total, size=n_sample, replace=False)
                df_sub = df_all_with_mob.iloc[idx].copy()

                df_sub_lab, _ = classify_dataframe(df_sub)

                valid = df_sub_lab["mobility"].notna()
                if valid.sum() == 0:
                    continue

                weighted_mob = df_sub_lab.loc[valid, "mobility"].mean()

                rows.append({
                    "n_sample": n_sample,
                    "bootstrap": boot,
                    "weighted_mobility": weighted_mob,
                })

    elif {"label_primary", "mobility"}.issubset(mob.columns):
        print("Using per-class mobility file.")

        class_mob = dict(zip(mob["label_primary"], mob["mobility"]))

        for n_sample in n_values:
            print(f"Mobility convergence N = {n_sample}")

            for boot in range(N_BOOT):
                # Reconstruct the same idea: sample, classify, compute class fractions
                idx = rng.choice(n_total, size=n_sample, replace=False)
                df_sub = df_all.iloc[idx].copy()

                df_sub_lab, _ = classify_dataframe(df_sub)
                frac = df_sub_lab["label_primary"].value_counts(normalize=True)

                weighted_mob = 0.0
                for label, f in frac.items():
                    if label in class_mob:
                        weighted_mob += f * class_mob[label]

                rows.append({
                    "n_sample": n_sample,
                    "bootstrap": boot,
                    "weighted_mobility": weighted_mob,
                })

    else:
        raise ValueError(
            "MOBILITY_FILE must contain either columns:\n"
            "  path, mobility\n"
            "or\n"
            "  label_primary, mobility"
        )

    df_mob_conv = pd.DataFrame(rows)

    if len(df_mob_conv) == 0:
        print("No mobility convergence data generated.")
        return

    out_csv = OUTPUT_DIR / "weighted_mobility_convergence_results.csv"
    df_mob_conv.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

    summary = (
        df_mob_conv.groupby("n_sample")["weighted_mobility"]
        .agg(
            mean="mean",
            q025=lambda x: np.percentile(x, 2.5),
            q975=lambda x: np.percentile(x, 97.5),
            std="std",
        )
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(7.2, 4.2))

    ax.plot(summary["n_sample"], summary["mean"], linewidth=2)
    ax.fill_between(summary["n_sample"], summary["q025"], summary["q975"], alpha=0.25)

    ax.set_xlabel("Number of sampled pathways")
    ax.set_ylabel(r"Class-weighted mobility")
    ax.set_title("Convergence of class-weighted mobility")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    savefig(fig, "MAIN_weighted_mobility_convergence")
    plt.show()

## 7. Threshold convergence (SI)

Tracks how the quantile-based classification thresholds themselves converge as more pathways are included.

In [ ]:
def plot_threshold_convergence(df_conv):
    summary = summarize_bootstrap(df_conv, "threshold")

    threshold_labels = [
        "pE_q_source",
        "pE_q_target",
        "E_dip_q_high",
        "C_dip_q_high",
        "rE_q_low",
        "Lweak_E_q_high",
        "drop_E_q_high",
        "rise_E_q_high",
    ]

    threshold_labels = [x for x in threshold_labels if x in summary["label"].unique()]

    fig, axes = plt.subplots(
        len(threshold_labels),
        1,
        figsize=(7.2, 2.0 * len(threshold_labels)),
        sharex=True,
    )

    if len(threshold_labels) == 1:
        axes = [axes]

    for ax, lab in zip(axes, threshold_labels):
        sub = summary[summary["label"] == lab]

        ax.plot(sub["n_sample"], sub["mean"], linewidth=2)
        ax.fill_between(sub["n_sample"], sub["q025"], sub["q975"], alpha=0.25)

        ax.set_ylabel(lab)
        ax.grid(alpha=0.3)

    axes[-1].set_xlabel("Number of sampled pathways")

    fig.suptitle("Convergence of quantile-based classification thresholds", y=1.01)
    plt.tight_layout()

    savefig(fig, "SI_threshold_convergence")
    plt.show()

## 8. Bootstrap uncertainty (SI)

95% bootstrap interval width of each class fraction as a function of sample size.

In [ ]:
def plot_bootstrap_uncertainty(df_conv, labels):
    summary = summarize_bootstrap(df_conv, "primary_fraction")

    fig, ax = plt.subplots(figsize=(7.2, 4.4))

    for lab in labels:
        sub = summary[summary["label"] == lab]
        if len(sub) == 0:
            continue

        ci_width = sub["q975"] - sub["q025"]

        ax.plot(
            sub["n_sample"],
            ci_width,
            linewidth=2,
            label=lab
        )

    ax.set_xlabel("Number of sampled pathways")
    ax.set_ylabel("95% bootstrap interval width")
    ax.set_title("Bootstrap uncertainty of class fractions")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)

    plt.tight_layout()
    savefig(fig, "SI_bootstrap_label_uncertainty")
    plt.show()

## 9. Threshold sensitivity (SI)

Re-classifies the full dataset under low/default/high quantile cutoffs and reports label agreement with the default classification.

In [ ]:
def run_threshold_sensitivity():
    """
    Test whether qualitative class fractions are stable when
    quantile thresholds are slightly changed.
    """

    df_all = pd.read_csv(ORDER_DESCRIPTOR_FILE)

    settings = [
        {
            "name": "low_cutoff",
            "q_source": 15,
            "q_target": 85,
            "q_high": 80,
            "q_low": 20,
        },
        {
            "name": "default",
            "q_source": 20,
            "q_target": 80,
            "q_high": 75,
            "q_low": 25,
        },
        {
            "name": "high_cutoff",
            "q_source": 25,
            "q_target": 75,
            "q_high": 70,
            "q_low": 30,
        },
    ]

    rows = []
    label_reference = None

    for setting in settings:
        name = setting["name"]

        df_lab, thresholds = classify_dataframe(
            df_all,
            q_source=setting["q_source"],
            q_target=setting["q_target"],
            q_high=setting["q_high"],
            q_low=setting["q_low"],
        )

        counts = df_lab["label_primary"].value_counts()
        frac = df_lab["label_primary"].value_counts(normalize=True)

        if name == "default":
            label_reference = df_lab[["path", "label_primary"]].copy()
            label_reference = label_reference.rename(columns={"label_primary": "label_default"})

        for lab, val in frac.items():
            rows.append({
                "setting": name,
                "label_primary": lab,
                "fraction": float(val),
                "count": int(counts[lab]),
                "q_source": setting["q_source"],
                "q_target": setting["q_target"],
                "q_high": setting["q_high"],
                "q_low": setting["q_low"],
            })

    df_sens = pd.DataFrame(rows)

    # Label agreement relative to default
    agreement_rows = []

    if label_reference is not None:
        for setting in settings:
            name = setting["name"]

            df_lab, _ = classify_dataframe(
                df_all,
                q_source=setting["q_source"],
                q_target=setting["q_target"],
                q_high=setting["q_high"],
                q_low=setting["q_low"],
            )

            tmp = df_lab[["path", "label_primary"]].merge(label_reference, on="path", how="left")
            agreement = np.mean(tmp["label_primary"] == tmp["label_default"])

            agreement_rows.append({
                "setting": name,
                "agreement_with_default": agreement,
            })

    df_agree = pd.DataFrame(agreement_rows)

    out_sens = OUTPUT_DIR / "threshold_sensitivity_class_fractions.csv"
    out_agree = OUTPUT_DIR / "threshold_sensitivity_label_agreement.csv"

    df_sens.to_csv(out_sens, index=False)
    df_agree.to_csv(out_agree, index=False)

    print(f"Saved:\n  {out_sens}\n  {out_agree}")

    return df_sens, df_agree


def plot_threshold_sensitivity(df_sens, df_agree):
    labels = ordered_present_labels(df_sens["label_primary"])

    settings = ["low_cutoff", "default", "high_cutoff"]

    x = np.arange(len(settings))
    width = 0.8 / max(1, len(labels))

    fig, axes = plt.subplots(2, 1, figsize=(7.2, 7.0), sharex=False)

    ax = axes[0]

    for i, lab in enumerate(labels):
        vals = []
        for setting in settings:
            sub = df_sens[
                (df_sens["setting"] == setting)
                & (df_sens["label_primary"] == lab)
            ]
            vals.append(float(sub["fraction"].iloc[0]) if len(sub) > 0 else 0.0)

        ax.bar(
            x + (i - len(labels) / 2) * width + width / 2,
            vals,
            width=width,
            label=lab,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(settings)
    ax.set_ylabel("Class fraction")
    ax.set_title("Sensitivity of class fractions to quantile thresholds")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(frameon=False, fontsize=8, ncol=2)

    ax = axes[1]

    ax.bar(
        df_agree["setting"],
        df_agree["agreement_with_default"],
        width=0.6,
    )

    ax.set_ylim(0, 1)
    ax.set_ylabel("Agreement with default labels")
    ax.set_title("Path-level label agreement under threshold perturbation")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    savefig(fig, "SI_threshold_sensitivity")
    plt.show()

## 10. Run all analyses

In [ ]:
print("\n========== MAIN FIGURE 1: CLASS FINGERPRINTS ==========")
plot_class_fingerprints()

print("\n========== MAIN FIGURE 2 + SI DATA: SUBSAMPLING CONVERGENCE ==========")
df_conv, df_full, labels, full_frac, full_gb_frac = run_subsampling_convergence()

print("\n========== MAIN FIGURE 2: SUBSAMPLING CONVERGENCE PLOT ==========")
plot_main_subsampling_convergence(df_conv, labels, full_frac, full_gb_frac)

print("\n========== MAIN OPTIONAL: WEIGHTED MOBILITY CONVERGENCE ==========")
plot_weighted_mobility_convergence(df_conv, df_full)

print("\n========== SI FIGURE 1: THRESHOLD CONVERGENCE ==========")
plot_threshold_convergence(df_conv)

print("\n========== SI FIGURE 2: BOOTSTRAP UNCERTAINTY ==========")
plot_bootstrap_uncertainty(df_conv, labels)

print("\n========== SI FIGURE 3: THRESHOLD SENSITIVITY ==========")
df_sens, df_agree = run_threshold_sensitivity()
plot_threshold_sensitivity(df_sens, df_agree)

print("\nAll figures saved to:")
print(OUTPUT_DIR)